# 06 - Single-file and batch audio prediction

This notebook demonstrates single-file and batch prediction using the model selected by validation macro-F1 and evaluated in notebook 05. No model training is performed in this stage, and the held-out test set is not used. The reported confidence scores are model scores rather than calibrated probabilities.

**The evaluation notebook must be completed before this stage.**

## 1. Imports and load the selected model

In [4]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.prediction import AudioCommandPredictor
from src.preprocessing import load_audio_segment

predictor = AudioCommandPredictor(PROJECT_ROOT)
print('Selected model:', predictor.model_name)
print('Recognised classes:', predictor.labels)


Selected model: small_audio_cnn
Recognised classes: ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'silence', 'stop', 'unknown', 'up', 'yes']


## 2. Single-file prediction

A WAV recording of at least one second is supplied for prediction. The audio is converted to mono, resampled to 16 kHz, and padded or trimmed to one second before the model is applied.

In [5]:
AUDIO_PATH = PROJECT_ROOT / 'data' / 'inference_samples' / 'yes.wav'
if AUDIO_PATH.is_file():
    single_result = predictor.predict_one(AUDIO_PATH)
    display(pd.DataFrame(single_result['top_predictions']))
    print('Predicted command:', single_result['predicted_command'])
    print('Confidence score:', round(single_result['confidence_score'], 4))
    print(single_result['confidence_note'])
else:
    print('Add a WAV file at:', AUDIO_PATH)


,label,score
0,yes,0.449953
1,left,0.196073
2,no,0.140726


Predicted command: yes
Confidence score: 0.45
Model score; not a calibrated probability.


## 3. Batch prediction

Multiple WAV recordings are placed in the inference folder and processed together. The resulting CSV contains one prediction row for each input file.

In [6]:
INFERENCE_DIR = PROJECT_ROOT / 'data' / 'inference_samples'
OUTPUT_CSV = PROJECT_ROOT / 'outputs' / 'batch_predictions.csv'
audio_files = sorted(INFERENCE_DIR.glob('*.wav')) if INFERENCE_DIR.is_dir() else []
if audio_files:
    batch_results = predictor.predict_files(audio_files, output_csv=OUTPUT_CSV)
    display(batch_results)
    print('Saved batch predictions to:', OUTPUT_CSV)
else:
    print('No WAV files found. Add WAV recordings to:', INFERENCE_DIR)


,file,model,predicted_command,confidence_score
0,no.wav,small_audio_cnn,no,0.639293
1,stop.wav,small_audio_cnn,stop,0.677242
2,up.wav,small_audio_cnn,up,0.507641
3,yes.wav,small_audio_cnn,yes,0.449953


Saved batch predictions to: c:\Users\a\Documents\machine_learning_cycle\outputs\batch_predictions.csv


## 4. Interpretation of prediction output

- `predicted_command` is one of the 12 trained labels: ten commands, `unknown` or `silence`.
- `confidence_score` is the model score for the predicted class. It can be used to rank predictions, but it is not calibrated confidence and should not be interpreted as a guaranteed probability.
- `top_predictions` contains the highest-scoring classes for a single recording.
- `batch_predictions.csv` stores the batch results and can be used by the API and dashboard.

The prediction module performs inference only. Model updates are handled separately through the controlled retraining workflow.